In [2]:
# ** This cell is needed since we are not in the src directory 
import sys 
import os
# Add the src/ directory to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../src/")))

ROOT_DIR = ".."
SRC_DIR = ROOT_DIR + "/src"

import sys
sys.path.append("/Users/admin/eeg-ds004504/")


In [3]:
from config_handler import initiate_config, load_config

initiate_config()

{'data_path': '/Users/admin/eeg-ds004504',
 'derivatives': False,
 'freqBands': {'Alpha': [8, 12],
  'Beta': [12, 30],
  'Delta': [0.5, 4],
  'Theta': [4, 8]},
 'method': 'welch',
 'stepSize': 1.5,
 'windowLength': 3}

In [4]:
print(load_config())

{'data_path': '/Users/admin/eeg-ds004504', 'derivatives': False, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8]}, 'method': 'welch', 'stepSize': 1.5, 'windowLength': 3}


In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType, MapType
import pandas as pd

In [6]:
# Check if there's an active Spark context and stop it
from pyspark import SparkContext
if SparkContext._active_spark_context:
    print("Stopping existing Spark context...")
    SparkContext._active_spark_context.stop()
    print("Previous Spark context stopped successfully")

In [7]:
# creating the ultimate most optimised spark session ever muwahaha
import os
from pyspark.sql import SparkSession

# Set Java options for the JVM running Spark
# -Xmx12g : Sets the maximum heap size to 12GB
# -Xms4g : Sets the initial heap size to 4GB to avoid resizing overhead
os.environ["_JAVA_OPTIONS"] = "-Xmx12g -Xms4g"

# Build Spark session with memory, parallelism, and network settings
spark = (
    SparkSession.builder 
    # Application name shown in Spark UI
    .appName("EEG_Analysis") 

    # Use all available logical cores or specify a number
    # "local[*]" uses all available cores, "local[12]" limits to 12 threads
    .config("spark.master", "local[12]") \

    # Executor memory: how much memory each Spark worker can use
    .config("spark.executor.memory", "8g") \

    # Driver memory: memory available to the Spark driver (main Python process)
    .config("spark.driver.memory", "8g") \

    # Number of shuffle partitions (e.g., after groupBy, join, etc.)
    # Lower this in local mode to reduce overhead (default is 200)
    .config("spark.sql.shuffle.partitions", "12") \

    # Default number of partitions in operations like parallelize
    .config("spark.default.parallelism", "12") \

    # Maximum size (in MB) allowed for any RPC message (e.g., large UDF closures or data broadcasts)
    .config("spark.rpc.message.maxSize", "256") \

    # Required for avoiding binding issues on some MacOS environments
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.host", "127.0.0.1") 
    .getOrCreate()
)

    # ----------------------------------------
    # Additional advanced options (optional):
    # ----------------------------------------

    # Use Kryo serializer instead of default Java serializer for better performance
    # .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \

    # Increase broadcast join timeout (in seconds) for large models or lookup tables
    # .config("spark.sql.broadcastTimeout", "600") \

    # Fraction of JVM memory reserved for execution and storage (default is 0.6)
    # .config("spark.memory.fraction", "0.8") \

    # Portion of memory reserved for caching/storage (default is 0.5 of memory.fraction)
    # .config("spark.memory.storageFraction", "0.3") \

    # Enable Apache Arrow for efficient pandas-to-Spark conversion (useful with UDFs)
    # .config("spark.sql.execution.arrow.pyspark.enabled", "true") \

    # Finalize and create the Spark session


spark = SparkSession.builder.appName("MyApp").getOrCreate()

print("New Spark session created successfully")

Picked up _JAVA_OPTIONS: -Xmx12g -Xms4g
Picked up _JAVA_OPTIONS: -Xmx12g -Xms4g
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/10 17:26:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/04/10 17:26:24 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


New Spark session created successfully


25/04/10 17:26:24 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [8]:
def reload_my_modules():
    import importlib
    import populate_schemas
    import feature_extraction
    import schema_definition
    importlib.reload(populate_schemas)
    importlib.reload(feature_extraction)
    importlib.reload(schema_definition)

reload_my_modules()


from populate_schemas import load_subjects_df, extract_features_udtf
from feature_extraction import processEpoch, processSub
from schema_definition import get_feature_schema, get_subject_schema

sc = spark.sparkContext # we pass udf/udtf's (user defind functions and user defined table functions) to spark so it can access them

# Making all necessary modules available to spark
try: 
    # oh btw spark is werid about not finding the config but it always finds it somehow not sure how that works not going to look rn tbh
    # ^ so feature_extraction has extra print statements
    sc.addPyFile(os.path.join(SRC_DIR, "feature_extraction.py"))
    print("Added feature_extraction.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "preprocess_sets.py"))
    print("Added preprocess_sets to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "schema_definition.py"))
    print("Added schema_definition.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "config_handler.py"))
    print("Added config_handler.py to the pyspark context")
except Exception as e:
    print(f"Error adding files to SparkContext: {e}")

Config not found in feature_extraction.py
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': False, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8]}, 'method': 'welch', 'stepSize': 1.5, 'windowLength': 3}
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': False, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8]}, 'method': 'welch', 'stepSize': 1.5, 'windowLength': 3}
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': False, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8]}, 'method': 'welch', 'stepSize': 1.5, 'windowLength': 3}
Added feature_extraction.py to the pyspark context
Added preprocess_sets to the pyspark context
Added schema_definition.py to the pyspark context
Added config_handler.py to the pyspark context


In [9]:
import pandas as pd

alz_df_pandas = pd.read_pickle("alz_df_apr10_1355.pkl")
cntrl_df_pandas = pd.read_pickle("cntrl_df_apr10_1355.pkl")


In [10]:
%%time
alz_df_spark = spark.createDataFrame(alz_df_pandas)
cntrl_df_spark = spark.createDataFrame(cntrl_df_pandas)

CPU times: user 1min 11s, sys: 544 ms, total: 1min 11s
Wall time: 1min 12s


In [11]:
#just renaming things now that we understand the types and where things are coming from
alz_df = alz_df_spark
cntrl_df = cntrl_df_spark

In [12]:
alz_df.show()

25/04/10 17:27:41 WARN TaskSetManager: Stage 0 contains a task of very large size (11840 KiB). The maximum recommended task size is 1000 KiB.
25/04/10 17:27:45 WARN PythonRunner: Detected deadlock while completing task 0.0 in stage 0 (TID 0): Attempting to kill Python Worker
                                                                                

+---------+-------+---------+--------+-----------+--------------------+----------+
|SubjectID|EpochID|Electrode|WaveBand|FeatureName|        FeatureValue|table_type|
+---------+-------+---------+--------+-----------+--------------------+----------+
|  sub-008|   ep-0|      Fp1|   Alpha|      Power|7.020328193902969E-4|      band|
|  sub-008|   ep-0|      Fp1|    Beta|      Power|3.399499109946191...|      band|
|  sub-008|   ep-0|      Fp1|   Delta|      Power| 0.08723169565200806|      band|
|  sub-008|   ep-0|      Fp1|   Theta|      Power|0.001139140920713544|      band|
|  sub-008|   ep-0|      Fp1|    NULL|TotalEnergy| 0.34805238246917725| electrode|
|  sub-008|   ep-0|      Fp1|    NULL| TotalPower| 0.01123595517128706| electrode|
|  sub-008|   ep-0|      Fp2|   Alpha|      Power|0.001468957751058042|      band|
|  sub-008|   ep-0|      Fp2|    Beta|      Power|5.043480778113008E-4|      band|
|  sub-008|   ep-0|      Fp2|   Delta|      Power| 0.08462297916412354|      band|
|  s

# Raw Data Visualization

# Start of data processing

In [36]:
# give each its respective labels
from pyspark.sql.functions import lit
alz_df = alz_df.withColumn("label", lit(1)).repartition(16).persist()
cntrl_df = cntrl_df.withColumn("label", lit(0)).repartition(16).persist()

In [37]:
# union everything
full_df = alz_df.unionByName(cntrl_df)

In [38]:
# Split based on feature type
from pyspark.sql.functions import col

band_df = full_df.filter(col("table_type") == "band")
channel_df = full_df.filter(col("table_type") == "electrode")
epoch_df = full_df.filter(col("table_type") == "epoch")


In [39]:
from pyspark.sql.functions import concat_ws

# Band-level: Electrode_WaveBand_Feature
band_df = band_df.withColumn("pivot", concat_ws("_", "Electrode", "WaveBand", "FeatureName"))

# Channel-level: Electrode_Feature
channel_df = channel_df.withColumn("pivot", concat_ws("_", "Electrode", "FeatureName"))

# Epoch-level: just FeatureName
epoch_df = epoch_df.withColumn("pivot", col("FeatureName"))


In [40]:
from pyspark.sql.functions import first

# Pivot band-level features
band_pivot = band_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))

# Pivot channel-level features
channel_pivot = channel_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))

# Pivot epoch-level features
epoch_pivot = epoch_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))

In [57]:
from functools import reduce

# full_df = reduce(
#     lambda df1, df2: df1.join(df2, on=["SubjectID", "EpochID", "label"], how="outer"),
#     [band_pivot, channel_pivot, epoch_pivot]
# ).fillna(0.0)


full_df = reduce(
    lambda df1, df2: df1.join(df2, on=["SubjectID", "EpochID", "label"], how="outer"),
     [epoch_pivot]# band_pivot, channel_pivot, epoch_pivot]
).fillna(0.0)
full_df.repartition(16).persist()


DataFrame[SubjectID: string, EpochID: string, label: int, AppEntropy: double, HiguchiFD: double, HjorthComplexity: double, HjorthMobility: double, KatzFD: double, Kurtosis: double, Mean: double, RMS: double, SampleEntropy: double, Skewness: double, Std: double, Variance: double]

In [58]:
type(full_df)

pyspark.sql.dataframe.DataFrame

In [59]:
NUM_TEST_SUBJECTS_PER_GROUP = 2

# Get test subject IDs from full_df (which has .label)
alz_test_subjects = (
    full_df.filter("label == 1")
    .select("SubjectID")
    .distinct()
    .orderBy("SubjectID")
    .limit(NUM_TEST_SUBJECTS_PER_GROUP)
    .rdd.flatMap(lambda row: row)
    .collect()
)

cntrl_test_subjects = (
    full_df.filter("label == 0")
    .select("SubjectID")
    .distinct()
    .orderBy("SubjectID")
    .limit(NUM_TEST_SUBJECTS_PER_GROUP)
    .rdd.flatMap(lambda row: row)
    .collect()
)

test_subjects = alz_test_subjects + cntrl_test_subjects #this will be the firs 2 subjects of each group for reproduceablility

In [60]:
# Split into test and train sets
test_df = full_df.filter(col("SubjectID").isin(test_subjects))
train_df = full_df.filter(~col("SubjectID").isin(test_subjects))


In [61]:
import dimensionality_reduction
import importlib
importlib.reload(dimensionality_reduction)
from dimensionality_reduction import min_max_normalize
feature_cols = [c for c in train_df.columns if c not in ("SubjectID", "EpochID", "label")]
train_norm_df, test_norm_df = min_max_normalize(train_df, test_df, feature_cols)

In [62]:
pca_input_cols = feature_cols
from dimensionality_reduction import fit_pca_model

pca_model, k_val = fit_pca_model(train_norm_df, pca_input_cols, variance_target=0.95)

print(f"PCA model fitted with {k_val} components to capture 95% variance")

PCA model fitted with 4 components to capture 95% variance


In [63]:
pca_model.explainedVariance

DenseVector([0.5446, 0.3577, 0.0343, 0.0208])

In [64]:
from dimensionality_reduction import apply_pca_model

train_df = apply_pca_model(train_norm_df, pca_input_cols, pca_model, k_val)
test_df = apply_pca_model(test_norm_df, pca_input_cols, pca_model, k_val)


# ML time

In [65]:
train_pd = train_df.toPandas()
test_pd = test_df.toPandas()


In [66]:
import numpy as np

# Convert Spark DenseVectors to regular 2D numpy arrays
X_train = np.array(train_pd["features"].tolist())
y_train = train_pd["label"].values

X_test = np.array(test_pd["features"].tolist())
y_test = test_pd["label"].values


In [67]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)


X_train shape: (33756, 4)
y_train shape: (33756,)


In [68]:
y_train

array([1, 1, 1, ..., 1, 0, 0], dtype=int32)

In [69]:
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
import numpy as np

# Define models
models = {
    "KNN": KNeighborsClassifier(),
    "SVM": make_pipeline(StandardScaler(), SVC(probability=True)),
    "NeuralNet": MLPClassifier(hidden_layer_sizes=(100,), max_iter=10000, random_state=42),
    "DecisionTree": DecisionTreeClassifier(
        max_depth=8,
        max_features=None,
        min_samples_leaf=10,
        min_samples_split=5,
        random_state=42
    ),
    "GradientBoostedTrees": GradientBoostingClassifier(n_estimators=100, random_state=42)
}

from sklearn.model_selection import StratifiedKFold

# Run 15-fold CV with parallelization
for name, model in models.items():
    print(f"\n=== Cross-Validation: {name} ===")
    
    # Cross-validation scores
    scores = cross_val_score(model, X_train, y_train, cv=15, scoring='accuracy', n_jobs=-1)
    
    print(f"Mean Accuracy: {scores.mean():.4f}")
    print(f"Standard Deviation: {scores.std():.4f}")
    print(f"All Fold Scores: {np.round(scores, 4)}")
    
    # Get predictions from best-performing fold
    best_fold_index = np.argmax(scores)
    
    # Refit on best 14/15 folds and evaluate on the 1/15
    skf = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)
    for i, (train_index, test_index) in enumerate(skf.split(X_train, y_train)):
        if i == best_fold_index:
            X_tr, X_te = X_train[train_index], X_train[test_index]
            y_tr, y_te = y_train[train_index], y_train[test_index]
            
            model.fit(X_tr, y_tr)
            y_pred_test = model.predict(X_te)
            y_pred_train = model.predict(X_tr)

            test_acc = accuracy_score(y_te, y_pred_test)
            train_acc = accuracy_score(y_tr, y_pred_train)

            print(f"\n=== Best Fold Summary: {name} ===")
            print(f"Train Accuracy: {train_acc:.4f}")
            print(f"Test Accuracy: {test_acc:.4f}")
            print(classification_report(y_te, y_pred_test, target_names=["Control", "Alzheimer's"]))
            break



=== Cross-Validation: KNN ===
Mean Accuracy: 0.6764
Standard Deviation: 0.0115
All Fold Scores: [0.6735 0.6828 0.6828 0.6713 0.6766 0.6815 0.6609 0.6773 0.6627 0.6636
 0.6764 0.692  0.7036 0.6609 0.6796]

=== Best Fold Summary: KNN ===
Train Accuracy: 0.7814
Test Accuracy: 0.6787
              precision    recall  f1-score   support

     Control       0.65      0.63      0.64      1009
 Alzheimer's       0.70      0.72      0.71      1241

    accuracy                           0.68      2250
   macro avg       0.67      0.67      0.67      2250
weighted avg       0.68      0.68      0.68      2250


=== Cross-Validation: SVM ===
Mean Accuracy: 0.6855
Standard Deviation: 0.0100
All Fold Scores: [0.6757 0.6819 0.6961 0.6721 0.6904 0.6704 0.6889 0.6764 0.6751 0.6956
 0.6902 0.7018 0.7013 0.684  0.6822]

=== Best Fold Summary: SVM ===
Train Accuracy: 0.6862
Test Accuracy: 0.6933
              precision    recall  f1-score   support

     Control       0.73      0.51      0.60      1009


In [30]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [34]:


models = {
    "KNN": KNeighborsClassifier(),
    "SVM": make_pipeline(StandardScaler(), SVC(probability=True)),
    "NeuralNet": MLPClassifier(hidden_layer_sizes=(100,), max_iter=10000, random_state=42),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "GradientBoostedTrees": GradientBoostingClassifier(n_estimators=100, random_state=42)
}

for name, model in models.items():
    print(f"\n=== Model: {name} ===")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred, target_names=["Control", "Alzheimer's"]))



=== Model: KNN ===
Accuracy: 0.6647
              precision    recall  f1-score   support

     Control       0.69      0.70      0.70      1112
 Alzheimer's       0.63      0.62      0.63       925

    accuracy                           0.66      2037
   macro avg       0.66      0.66      0.66      2037
weighted avg       0.66      0.66      0.66      2037


=== Model: SVM ===


KeyboardInterrupt: 

In [33]:
from sklearn.model_selection import cross_val_score
import numpy as np

# Define models again
models = {
    "KNN": KNeighborsClassifier(),
    "SVM": make_pipeline(StandardScaler(), SVC(probability=True)),
    "NeuralNet": MLPClassifier(hidden_layer_sizes=(100,), max_iter=10000, random_state=42),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "GradientBoostedTrees": GradientBoostingClassifier(n_estimators=100, random_state=42)
}

# Run 15-fold cross-validation for each model
for name, model in models.items():
    print(f"\n=== Cross-Validation: {name} ===")
    scores = cross_val_score(model, X_train, y_train, cv=15, scoring='accuracy')
    print(f"Mean Accuracy: {scores.mean():.4f}")
    print(f"Standard Deviation: {scores.std():.4f}")
    print(f"All Fold Scores: {np.round(scores, 4)}")



=== Cross-Validation: KNN ===
Mean Accuracy: 0.7449
Standard Deviation: 0.0085
All Fold Scores: [0.745  0.7557 0.7401 0.737  0.7508 0.7481 0.7444 0.7569 0.7529 0.7427
 0.7453 0.7573 0.7333 0.7311 0.7333]

=== Cross-Validation: SVM ===


KeyboardInterrupt: 

In [52]:
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.metrics import classification_report, accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
import numpy as np

# Define models
models = {
    "KNN": KNeighborsClassifier(),
    "SVM": make_pipeline(StandardScaler(), SVC(probability=True)),
    "NeuralNet": MLPClassifier(hidden_layer_sizes=(100,), max_iter=10000, random_state=42),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "GradientBoostedTrees": GradientBoostingClassifier(n_estimators=100, random_state=42)
}

# Run 15-fold CV with parallelization
for name, model in models.items():
    print(f"\n=== Cross-Validation: {name} ===")
    
    # Cross-validation scores
    scores = cross_val_score(model, X_train, y_train, cv=15, scoring='accuracy', n_jobs=-1)
    
    print(f"Mean Accuracy: {scores.mean():.4f}")
    print(f"Standard Deviation: {scores.std():.4f}")
    print(f"All Fold Scores: {np.round(scores, 4)}")
    
    # Get predictions from best-performing fold
    best_fold_index = np.argmax(scores)
    
    # To simulate best fold's report, refit on best 14/15 folds and evaluate on 1/15
    from sklearn.model_selection import StratifiedKFold
    skf = StratifiedKFold(n_splits=15, shuffle=True, random_state=42)
    for i, (train_index, test_index) in enumerate(skf.split(X_train, y_train)):
        if i == best_fold_index:
            X_tr, X_te = X_train[train_index], X_train[test_index]
            y_tr, y_te = y_train[train_index], y_train[test_index]
            model.fit(X_tr, y_tr)
            y_pred = model.predict(X_te)
            acc = accuracy_score(y_te, y_pred)
            print(f"\n=== Best Fold Summary: {name} ===")
            print(f"Accuracy: {acc:.4f}")
            print(classification_report(y_te, y_pred, target_names=["Control", "Alzheimer's"]))
            break



=== Cross-Validation: KNN ===
Mean Accuracy: 0.5944
Standard Deviation: 0.0084
All Fold Scores: [0.5877 0.5971 0.5917 0.6046 0.6104 0.5966 0.5987 0.596  0.5773 0.5924
 0.5956 0.5933 0.584  0.6058 0.5853]

=== Best Fold Summary: KNN ===
Accuracy: 0.5793
              precision    recall  f1-score   support

     Control       0.53      0.50      0.52      1009
 Alzheimer's       0.61      0.64      0.63      1242

    accuracy                           0.58      2251
   macro avg       0.57      0.57      0.57      2251
weighted avg       0.58      0.58      0.58      2251


=== Cross-Validation: SVM ===
Mean Accuracy: 0.6465
Standard Deviation: 0.0059
All Fold Scores: [0.6419 0.6495 0.6335 0.6459 0.6473 0.6579 0.652  0.6484 0.6409 0.6436
 0.6542 0.6409 0.6453 0.6453 0.6507]

=== Best Fold Summary: SVM ===
Accuracy: 0.6375
              precision    recall  f1-score   support

     Control       0.65      0.42      0.51      1009
 Alzheimer's       0.63      0.82      0.71      1242

 